# Finetuning Mask R-CNN

based on this tutorial
https://pytorch.org/tutorials/intermediate/torchvision_tutorial.html


In [1]:
import os

import torch
from torchvision import tv_tensors
from torchvision.io import read_image
from torchvision.transforms.v2 import functional as F

import cv2
import numpy as np


COLOR_MAP = {
    51: 1,
    102: 2,
    153: 3,
    204: 4,
    255: 5,
}

# our dataset has two classes only - background and colony
num_classes = len(COLOR_MAP) + 1


class ColonySegmentationDataset(torch.utils.data.Dataset):
    def __init__(self, root, transforms):
        self.root = root
        self.transforms = transforms

        # load all image files, sorting them to ensure that they are aligned
        self.imgs = list(sorted(os.listdir(os.path.join(root, "images"))))
        self.masks = list(sorted(os.listdir(os.path.join(root, "class_masks"))))

    def __getitem__(self, idx):
        img_path = os.path.join(self.root, "images", self.imgs[idx])
        mask_path = os.path.join(self.root, "class_masks", self.masks[idx])
        img = read_image(img_path)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask)

        masks = np.zeros((num_labels - 1, *mask.shape), dtype=np.uint8)
        class_labels = np.zeros(num_labels - 1, dtype=np.uint8)

        bboxes = np.zeros((num_labels - 1, 4), dtype=np.float32)
        for i, stat in enumerate(stats[1:]):
            x, y, w, h = stat[:4]
            bboxes[i] = [x, y, x + w, y + h]

        for i in range(1, num_labels):
            # extract the mask for the current connected component
            region_mask = labels == i
            masks[i - 1] = region_mask

            color = np.median(mask[region_mask])
            class_labels[i - 1] = COLOR_MAP[color]

        bboxes_torch = torch.tensor(bboxes, dtype=torch.float32)

        assert len(masks) == len(class_labels)

        masks = torch.tensor(masks, dtype=torch.uint8)

        img = tv_tensors.Image(img)

        target = {
            "boxes": tv_tensors.BoundingBoxes(
                bboxes_torch, format="XYXY", canvas_size=F.get_size(img)
            ),
            "masks": tv_tensors.Mask(masks),
            # there is only one class (colony)
            "labels": torch.tensor(class_labels, dtype=torch.int64),
            "image_id": idx,
            "iscrowd": torch.zeros(len(masks), dtype=torch.int64),
            "area": (bboxes_torch[:, 3] - bboxes_torch[:, 1])
            * (bboxes_torch[:, 2] - bboxes_torch[:, 0]),
        }

        if self.transforms is not None:
            img, target = self.transforms(img, target)

        return img, target

    def __len__(self):
        return len(self.imgs)

In [2]:
import torchvision
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor


def get_model_instance_segmentation(num_classes):
    # load an instance segmentation model pre-trained on COCO
    model = torchvision.models.detection.maskrcnn_resnet50_fpn_v2(
        weights="DEFAULT", box_detections_per_image=500
    )

    # get number of input features for the classifier
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # now get the number of input features for the mask classifier
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    hidden_layer = 256
    # and replace the mask predictor with a new one
    model.roi_heads.mask_predictor = MaskRCNNPredictor(
        in_features_mask, hidden_layer, num_classes
    )

    return model

In [3]:
from torchvision.transforms import v2 as T


def get_transforms(is_train: bool):
    """Get the transforms for the dataset.

    Args:
        is_train (bool): Whether the transforms are for training or testing.

    Returns:
        func: A function that takes in an image and target and applies the transforms.
    """

    transforms = []

    if is_train:
        transforms.append(T.RandomHorizontalFlip(0.5))
        # transforms.append(T.RandomVerticalFlip(0.5))
        # Rotation by 0, 90, 180, or 270 degrees
        # transforms.append(
        #     T.RandomChoice(
        #         [
        #             T.RandomRotation((0, 0)),
        #             T.RandomRotation((90, 90)),
        #             T.RandomRotation((180, 180)),
        #             T.RandomRotation((270, 270)),
        #         ]
        #     )
        # )
        # transforms.append(T.ColorJitter(brightness=0.1, contrast=0.1))

    transforms.append(T.ToDtype(torch.float, scale=True))
    transforms.append(T.ToPureTensor())
    return T.Compose(transforms)

## Testing `forward()` method

Before iterating over the dataset, it's good to see what the model expects
during training and inference time on sample data.


In [4]:
import utils

model = get_model_instance_segmentation(len(COLOR_MAP) + 1)
dataset = ColonySegmentationDataset(
    "../../training_data", get_transforms(is_train=True)
)
data_loader = torch.utils.data.DataLoader(
    dataset, batch_size=2, shuffle=True, collate_fn=utils.collate_fn
)

# For Training
images, targets = next(iter(data_loader))
images = list(image for image in images)
targets = [{k: v for k, v in t.items()} for t in targets]
output = model(images, targets)  # Returns losses and detections
print(output)

# For inference
model.eval()
x = [torch.rand(3, 300, 400), torch.rand(3, 500, 400)]
predictions = model(x)  # Returns predictions
print(predictions[0])

{'loss_classifier': tensor(1.6809, grad_fn=<NllLossBackward0>), 'loss_box_reg': tensor(0.2494, grad_fn=<DivBackward0>), 'loss_mask': tensor(1.0163, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>), 'loss_objectness': tensor(0.0854, grad_fn=<BinaryCrossEntropyWithLogitsBackward0>), 'loss_rpn_box_reg': tensor(0.0042, grad_fn=<DivBackward0>)}
{'boxes': tensor([[5.8883e+00, 0.0000e+00, 3.3321e+02, 3.0000e+02],
        [4.2464e-01, 2.0734e+02, 1.3743e+00, 3.0000e+02],
        [3.0073e+00, 8.4039e+01, 1.0006e+02, 2.0723e+02],
        [1.5397e+02, 3.0509e+00, 2.4065e+02, 1.2059e+01],
        [3.1141e+01, 1.6225e+00, 9.7331e+01, 1.4111e+01],
        [1.8881e+02, 2.7458e+00, 2.6481e+02, 1.2858e+01],
        [1.9136e+02, 0.0000e+00, 2.1187e+02, 2.2384e+01],
        [1.8011e+02, 5.2359e+01, 2.5228e+02, 2.4771e+02],
        [2.3386e+01, 1.3281e-01, 3.3173e+01, 1.6709e+01],
        [0.0000e+00, 1.4340e+02, 8.9808e+01, 2.6605e+02],
        [2.7586e+02, 0.0000e+00, 3.8390e+02, 1.5562e+02],
        [2

## Training and Validation


In [4]:
# train on the GPU or on the CPU, if a GPU is not available
device = torch.device("cpu")

# Check for accelerator availability
if torch.cuda.is_available():
    device = torch.device("cuda")
    print("Using CUDA for training.")
else:
    print("Using CPU for training.")

Using CPU for training.


In [16]:
from engine import evaluate, train_one_epoch


# use our dataset and defined transformations
dataset = ColonySegmentationDataset(
    "../../training_data", get_transforms(is_train=True)
)
dataset_test = ColonySegmentationDataset(
    "../../training_data", get_transforms(is_train=False)
)

# split the dataset in train and test set (last 30 images are test set)
indices = torch.randperm(len(dataset)).tolist()
dataset = torch.utils.data.Subset(dataset, indices[:-25])
dataset_test = torch.utils.data.Subset(dataset_test, indices[-25:])

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    collate_fn=utils.collate_fn,
)

data_loader_test = torch.utils.data.DataLoader(
    dataset_test,
    batch_size=1,
    shuffle=False,
    collate_fn=utils.collate_fn,
)

model = get_model_instance_segmentation(num_classes)
model.to(device)

params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(params, lr=0.005, momentum=0.9, weight_decay=0.0005)

lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)

num_epochs = 2

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()
    # evaluate on the test dataset
    evaluate(model, data_loader_test, device=device)

print("That's it!")

# Save the model to disk
torch.save(model.state_dict(), "model.pth")

Epoch: [0]  [ 0/22]  eta: 0:10:10  lr: 0.000243  loss: 4.0211 (4.0211)  loss_classifier: 1.5306 (1.5306)  loss_box_reg: 0.6357 (0.6357)  loss_mask: 0.7301 (0.7301)  loss_objectness: 1.0236 (1.0236)  loss_rpn_box_reg: 0.1011 (0.1011)  time: 27.7709  data: 2.1067


KeyboardInterrupt: 

In [5]:
import torch
import gradio as gr
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks
from pathlib import Path

model = get_model_instance_segmentation(num_classes).to(device)
model.load_state_dict(torch.load("model.pth", map_location=device))

eval_transform = get_transforms(is_train=False)


def threshold_predictions(pred, confidence_threshold):
    # Create a mask for scores above the threshold
    score_mask = torch.tensor(pred["scores"]) > confidence_threshold

    # Filter the labels and boxes using the mask
    filtered_labels = [label for label, keep in zip(pred["labels"], score_mask) if keep]
    filtered_boxes = pred["boxes"][score_mask]

    return {
        "labels": filtered_labels,
        "scores": pred["scores"][score_mask],
        "boxes": filtered_boxes,
        "masks": pred["masks"][score_mask],
    }


def predict(image, confidence_threshold):
    # convert numpy image to tensor
    image = torch.tensor(image).permute(2, 0, 1).float() / 255.0

    model.eval()
    with torch.no_grad():
        print(type(image))
        x = eval_transform(image)
        print(type(x))
        # convert RGBA -> RGB and move to device
        x = x[:3, ...].to(device)
        predictions = model([x])
        pred = predictions[0]

    image = (255.0 * (image - image.min()) / (image.max() - image.min())).to(
        torch.uint8
    )
    image = image[:3, ...]

    pred = threshold_predictions(pred, confidence_threshold)

    box_labels = [
        f"{label}:{score:.3f}" for label, score in zip(pred["labels"], pred["scores"])
    ]

    boxes = pred["boxes"].long()
    output_image = draw_segmentation_masks(
        output_image, masks, alpha=0.5, colors="yellow"
    )
    output_image = draw_bounding_boxes(
        image,
        boxes,
        box_labels,
        colors="#ee0044",
        width=4,
        font_size=32,
        font=str(Path("../../assets/zed-mono-bold.ttf")),
    )

    masks = (pred["masks"] > 0.7).squeeze(1)

    np_image = output_image.permute(1, 2, 0).numpy()
    total_colonies = sum([int(label) for label in pred["labels"]])

    return np_image, total_colonies


demo = gr.Interface(
    fn=predict,
    inputs=[
        "image",
        gr.Slider(label="Confidence threshold", maximum=1, value=0.9),
    ],
    outputs=[
        gr.Image(label="Segmented image"),
        gr.Number(label="Number of colonies detected"),
    ],
    title="Colony Detection",
    allow_flagging=False,
)

demo.launch(inline=False)

/opt/miniconda3/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running on local URL:  http://127.0.0.1:7860

To create a public link, set `share=True` in `launch()`.


<class 'torch.Tensor'>
<class 'torch.Tensor'>


/var/folders/pc/dr74t21x26b00xg9cb8dtsp80000gn/T/ipykernel_95550/136554669.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  score_mask = torch.tensor(pred["scores"]) > confidence_threshold


<class 'torch.Tensor'>
<class 'torch.Tensor'>


/var/folders/pc/dr74t21x26b00xg9cb8dtsp80000gn/T/ipykernel_95550/136554669.py:32: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  score_mask = torch.tensor(pred["scores"]) > confidence_threshold
